# 07 - EfficientNet-B0 at 320×320 with Subject-Grouped Cross-Validation

This notebook performs five-fold subject-grouped cross-validation on foreground-cropped, CLAHE-enhanced fingerprint images created directly from the original high-resolution PNGs.

**The locked holdout NPZ is never opened in this notebook.** Model design is based only on out-of-fold validation predictions.

In [ ]:
!pip install -q pandas numpy matplotlib scikit-learn torch torchvision tqdm

In [ ]:
from pathlib import Path
import gc, json, random, time, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, classification_report, f1_score, ConfusionMatrixDisplay
from google.colab import drive

SEED = 20260725
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda": raise RuntimeError("Select a T4 GPU in Runtime settings.")
print("GPU:", torch.cuda.get_device_name(0))

## Load only the cross-validation dataset

In [ ]:
drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/dermatoglyphic_project")
PACKAGE = PROJECT / "roll_efficientnet_320_package.zip"
WORK = Path("/content/efficientnet_320_cv")
DATA = WORK / "data"
MODELS = PROJECT / "models" / "efficientnet_320_cv"
FIGURES = PROJECT / "figures"
for path in (DATA, MODELS, FIGURES): path.mkdir(parents=True, exist_ok=True)

required = {"roll_320_clahe_cv.npz", "roll_320_clahe_cv_metadata.csv",
            "roll_320_clahe_locked_holdout.npz", "roll_320_clahe_locked_holdout_metadata.csv",
            "label_mapping.json", "README.txt"}
with zipfile.ZipFile(PACKAGE) as archive:
    assert archive.testzip() is None
    assert required.issubset(archive.namelist())
    # Security boundary: extract ONLY CV files. The locked holdout remains unopened.
    archive.extract("roll_320_clahe_cv.npz", DATA)
    archive.extract("roll_320_clahe_cv_metadata.csv", DATA)

assert not (DATA / "roll_320_clahe_locked_holdout.npz").exists()
with np.load(DATA / "roll_320_clahe_cv.npz", allow_pickle=False) as prepared:
    images = prepared["images"]
    labels = prepared["labels"]
    label_names = prepared["label_names"].tolist()
metadata = pd.read_csv(DATA / "roll_320_clahe_cv_metadata.csv", dtype={"subject_id": "string"})
assert images.shape == (1281, 320, 320) and images.dtype == np.uint8
assert len(labels) == len(metadata) == 1281
assert metadata["subject_id"].nunique() == 110
assert metadata["experiment_role"].eq("cross_validation").all()
assert np.array_equal(labels, metadata["label_id"].to_numpy(np.int64))
assert metadata["png_path"].is_unique and not metadata.isna().any().any()
print("CV-only security and integrity checks passed.")
print(images.shape, metadata["subject_id"].nunique(), label_names)

## Transforms

Augmentation is training-only. Horizontal flips are excluded because they can change loop orientation.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.ToPILImage(), transforms.RandomRotation(5),
    transforms.RandomAffine(0, translate=(0.025, 0.025), scale=(0.96, 1.04)),
    transforms.ColorJitter(brightness=0.08, contrast=0.12),
    transforms.Grayscale(3), transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.ToPILImage(), transforms.Grayscale(3),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
class FingerprintDataset(Dataset):
    def __init__(self, indices, transform): self.indices, self.transform = np.asarray(indices), transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, item):
        i = int(self.indices[item])
        return self.transform(images[i]), int(labels[i]), i

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for axis, i in zip(axes.ravel(), range(10)):
    axis.imshow(images[i], cmap="gray"); axis.set_title(label_names[labels[i]]); axis.axis("off")
plt.tight_layout(); plt.show()

## Training utilities

In [ ]:
def make_model():
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    model.classifier = nn.Sequential(nn.Dropout(0.35), nn.Linear(model.classifier[1].in_features, len(label_names)))
    return model.to(device)

def metrics(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro")}

def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None; model.train(training)
    loss_sum, truth, pred, row_indices = 0.0, [], [], []
    for x, y, idx in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(x); loss = criterion(logits, y)
            if training:
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        loss_sum += loss.item() * len(y)
        truth.extend(y.detach().cpu().numpy()); pred.extend(logits.argmax(1).detach().cpu().numpy())
        row_indices.extend(idx.numpy())
    return loss_sum / len(loader.dataset), np.asarray(truth), np.asarray(pred), np.asarray(row_indices)

def train_fold(fold, train_idx, val_idx):
    seed_everything(SEED + fold)
    train_subjects = set(metadata.iloc[train_idx]["subject_id"])
    val_subjects = set(metadata.iloc[val_idx]["subject_id"])
    assert train_subjects.isdisjoint(val_subjects)
    generator = torch.Generator().manual_seed(SEED + fold)
    train_loader = DataLoader(FingerprintDataset(train_idx, train_transform), 24, shuffle=True,
        num_workers=2, pin_memory=True, persistent_workers=True, generator=generator)
    val_loader = DataLoader(FingerprintDataset(val_idx, eval_transform), 32, shuffle=False,
        num_workers=2, pin_memory=True, persistent_workers=True)
    counts = np.bincount(labels[train_idx], minlength=len(label_names))
    weights = torch.tensor(np.sqrt(len(train_idx)/(len(label_names)*counts)), dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)
    model = make_model(); scaler = torch.amp.GradScaler("cuda")
    best_f1, stale, history = -1.0, 0, []
    best_path = MODELS / f"efficientnet_b0_320_fold_{fold}.pt"

    for p in model.features.parameters(): p.requires_grad = False
    stages = [("head", 3, torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4), 3)]
    for stage, epochs, optimizer, patience in stages:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=.3, patience=1)
        for epoch in range(1, epochs+1):
            start=time.perf_counter(); tr_loss, tr_y, tr_p, _=run_epoch(model,train_loader,criterion,optimizer,scaler)
            va_loss, va_y, va_p, va_i=run_epoch(model,val_loader,criterion); va=metrics(va_y,va_p); scheduler.step(va['macro_f1'])
            history.append({"fold":fold,"stage":stage,"epoch":epoch,"seconds":time.perf_counter()-start,
                "train_loss":tr_loss,"train_macro_f1":metrics(tr_y,tr_p)['macro_f1'],
                "validation_loss":va_loss,**{f"validation_{k}":v for k,v in va.items()}})
            if va['macro_f1'] > best_f1 + .001:
                best_f1=va['macro_f1']; stale=0; torch.save(model.state_dict(),best_path)
            else: stale+=1
            print(f"Fold {fold} {stage} {epoch}: val_f1={va['macro_f1']:.3f}, {history[-1]['seconds']:.1f}s")

    for p in model.parameters(): p.requires_grad = True
    optimizer=torch.optim.AdamW(model.parameters(),lr=7e-5,weight_decay=1e-4)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="max",factor=.3,patience=2)
    stale=0
    for epoch in range(1,16):
        start=time.perf_counter(); tr_loss,tr_y,tr_p,_=run_epoch(model,train_loader,criterion,optimizer,scaler)
        va_loss,va_y,va_p,va_i=run_epoch(model,val_loader,criterion); va=metrics(va_y,va_p); scheduler.step(va['macro_f1'])
        history.append({"fold":fold,"stage":"full","epoch":epoch,"seconds":time.perf_counter()-start,
            "train_loss":tr_loss,"train_macro_f1":metrics(tr_y,tr_p)['macro_f1'],
            "validation_loss":va_loss,**{f"validation_{k}":v for k,v in va.items()}})
        if va['macro_f1'] > best_f1 + .001:
            best_f1=va['macro_f1']; stale=0; torch.save(model.state_dict(),best_path)
        else: stale+=1
        print(f"Fold {fold} full {epoch}: val_f1={va['macro_f1']:.3f}, {history[-1]['seconds']:.1f}s")
        if stale >= 5: print("Early stop"); break
    model.load_state_dict(torch.load(best_path,map_location=device,weights_only=True))
    _,va_y,va_p,va_i=run_epoch(model,val_loader,criterion)
    del model, train_loader, val_loader; gc.collect(); torch.cuda.empty_cache()
    return history, va_i, va_y, va_p, best_f1

## Five-fold grouped cross-validation

Every subject appears in validation exactly once. No fold shares a subject between training and validation.

In [ ]:
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
groups = metadata["subject_id"].to_numpy()
oof_pred = np.full(len(labels), -1, dtype=np.int64)
oof_fold = np.full(len(labels), -1, dtype=np.int64)
all_history, fold_summary = [], []
for fold, (train_idx, val_idx) in enumerate(splitter.split(images, labels, groups), 1):
    print(f"\n===== FOLD {fold}: {len(train_idx)} train / {len(val_idx)} validation =====")
    assert set(groups[train_idx]).isdisjoint(set(groups[val_idx]))
    history, row_idx, truth, pred, best_f1 = train_fold(fold, train_idx, val_idx)
    assert np.array_equal(labels[row_idx], truth)
    oof_pred[row_idx] = pred; oof_fold[row_idx] = fold; all_history.extend(history)
    fold_summary.append({"fold":fold,"subjects":len(set(groups[val_idx])),"rows":len(val_idx),
        "accuracy":accuracy_score(truth,pred),"macro_f1":f1_score(truth,pred,average="macro"),
        "best_epoch_validation_macro_f1":best_f1})
assert (oof_pred >= 0).all() and (oof_fold >= 1).all()
history_table=pd.DataFrame(all_history); summary=pd.DataFrame(fold_summary)
history_table.to_csv(PROJECT/"efficientnet_320_cv_training_history.csv",index=False)
summary.to_csv(PROJECT/"efficientnet_320_cv_fold_summary.csv",index=False)
display(summary.round(3)); print("Mean fold macro F1:",summary.macro_f1.mean(),"SD:",summary.macro_f1.std())

## Out-of-fold results (the only model-selection result)

In [ ]:
print("OOF ACCURACY:",round(accuracy_score(labels,oof_pred),4))
print("OOF MACRO F1:",round(f1_score(labels,oof_pred,average="macro"),4))
report=pd.DataFrame(classification_report(labels,oof_pred,target_names=label_names,output_dict=True,zero_division=0)).T
display(report.round(3)); report.to_csv(PROJECT/"efficientnet_320_cv_oof_classification_report.csv")
predictions=metadata.copy(); predictions["cv_fold"]=oof_fold
predictions["true_label"]=[label_names[i] for i in labels]
predictions["predicted_label"]=[label_names[i] for i in oof_pred]
predictions["correct"]=predictions.true_label.eq(predictions.predicted_label)
predictions.to_csv(PROJECT/"efficientnet_320_cv_oof_predictions.csv",index=False)
fig,axis=plt.subplots(figsize=(7,6)); ConfusionMatrixDisplay.from_predictions(labels,oof_pred,
    display_labels=label_names,cmap="Blues",xticks_rotation=25,ax=axis,colorbar=False)
axis.set_title("EfficientNet-B0 320×320 Out-of-Fold Confusion Matrix")
plt.tight_layout(); plt.savefig(FIGURES/"efficientnet_320_cv_oof_confusion_matrix.png",dpi=160); plt.show()

## Stop here

Do not open `roll_320_clahe_locked_holdout.npz` yet. First compare this out-of-fold macro F1 and per-class performance with alternative preprocessing/model choices. Open the holdout exactly once only after the full modelling design is frozen.